# SANPO YOLO Gap Analysis & Fine-Tuning Assistant

**Purpose:** Identify which scene objects in SANPO are detected by **depth** but are completely missed by **YOLO26n** using its current COCO class whitelist.

This notebook serves as the primary workflow for evaluating YOLO performance on the SANPO dataset, mapping dataset gaps, saving hard examples, and automatically generating actionable fine-tuning recommendations.

### Key Capabilities:
1. **Centralized Configuration:** All thresholds, paths, and options in one place.
2. **GCS-Aware Streaming Loader:** Downloads/streams frames and depth maps directly from the GCP bucket anonymously when local assets are missing.
3. **Robust Blob Detection:** Morphological cleaning, Gaussian smoothing, and aspect ratio/solidity/extent calculation to isolate meaningful physical regions.
4. **Detailed Visualizations:** Produces a 5-panel layout (RGB, YOLO detections, depth heatmap, candidate regions, combined overlay).
5. **CSV Statistics Exports:** Generates `frame_summary.csv` and `candidate_regions.csv` for downstream analytics.
6. **Hard Example Mining:** Saves difficult/ambiguous frames automatically to `hard_examples/` for manual labeling.
7. **Automated Fine-Tuning Guidance:** Generates a structured markdown report prioritizing dataset improvements.

---
**Data structure assumed (if local):**
```
data/sanpo/raw/<session_hash>/camera_head/left/video_frames/000000.png
data/sanpo/raw/<session_hash>/camera_head/left/depth_maps/000000.float16.gz
```


In [ ]:
# ── Centralized Configuration ─────────────────────────────────────────────────
import sys
from pathlib import Path

# Add project root to path so we can import src modules directly
REPO_ROOT = Path("../").resolve()
sys.path.insert(0, str(REPO_ROOT))

# ── Paths & Model ─────────────────────────────────────────────────────────────
YOLO_MODEL_PATH = str(REPO_ROOT / "yolo26n.pt")
CONF_THRESH = 0.30
IOU_THRESH = 0.50
IMG_SIZE = (1280, 720)       # Size for model inference

# ── Depth Detection Parameters ────────────────────────────────────────────────
DEPTH_MIN_M = 0.5            # Min obstacle range in meters
DEPTH_MAX_M = 6.0            # Max obstacle range in meters
MIN_BLOB_AREA_PX = 800       # Minimum pixel size for a candidate region

# ── Stream / Session Selection ────────────────────────────────────────────────
# Set SESSION_FILTER = None to process all valid streams, or a list of session_ids.
SESSION_FILTER = None        # e.g., ["-5OCPnbrwJdu3jH70ieU7pUiFsOJQoeG"]
MAX_STREAMS = 3              # Process up to N streams to keep runtimes reasonable
MAX_FRAMES_PER_STREAM = 15   # Evenly sample up to N frames per stream
RANDOM_SEED = 42

# ── Outputs & Toggles ─────────────────────────────────────────────────────────
SAVE_VISUALIZATIONS = True   # Export 5-panel PNG visualizations for frames with gaps
EXPORT_CSV = True            # Export frame-level and region-level statistics to CSV
EXPORT_HARD_EXAMPLES = True  # Automatically export difficult frames to hard_examples/

OUTPUT_DIR = REPO_ROOT / "notebooks" / "gap_analysis_output"
HARD_EXAMPLES_DIR = REPO_ROOT / "notebooks" / "hard_examples"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HARD_EXAMPLES_DIR.mkdir(parents=True, exist_ok=True)

# Visualization displays in notebook
SHOW_PLOTS_IN_NOTEBOOK = True

print(f"✅ Centralized configuration loaded successfully.")
print(f"   - Model path:         {YOLO_MODEL_PATH}")
print(f"   - Output folder:      {OUTPUT_DIR}")
print(f"   - Hard examples:      {HARD_EXAMPLES_DIR}")


In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import os
import gzip
import json
import csv
import io
import time
import random
import numpy as np
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import ndimage
from pathlib import Path
from ultralytics import YOLO
import torch
from google.cloud import storage
from google.auth.credentials import AnonymousCredentials

print("✅ All imports successful.")


In [ ]:
# ── SANPO Loader & Retrieval Pipeline ──────────────────────────────────────────

class SANPOLoader:
    """
    Handles data retrieval for the SANPO dataset.
    Checks for local cached assets first; if unavailable, streams/downloads
    anonymously from the official Google Cloud Storage bucket.
    """
    def __init__(self, local_raw_path=None, camera="head", view="left"):
        self.camera = camera
        self.view = view
        self.local_raw_path = Path(local_raw_path) if local_raw_path else None
        
        # GCP Storage Details
        self.sanpo_root = "gs://gresearch/sanpo_dataset/v0/sanpo-real"
        self.client = storage.Client(credentials=AnonymousCredentials(), project='gresearch')
        parts = self.sanpo_root.split("/")
        self.bucket_name = parts[2]
        self.prefix = "/".join(parts[3:])
        self.bucket = self.client.bucket(self.bucket_name)

    def load_rgb(self, session_id: str, frame_id: int) -> np.ndarray | None:
        """Loads RGB frame either from local path or GCP bucket."""
        # 1. Try local first
        if self.local_raw_path:
            local_path = self.local_raw_path / session_id / f"camera_{self.camera}" / self.view / "video_frames" / f"{frame_id:06d}.png"
            if local_path.exists():
                img = cv2.imread(str(local_path))
                if img is not None:
                    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # 2. Try GCS anonymous download
        path = f"{self.prefix}/{session_id}/camera_{self.camera}/{self.view}/video_frames/{frame_id:06d}.png"
        try:
            blob = self.bucket.blob(path)
            arr = np.frombuffer(blob.download_as_bytes(timeout=10), np.uint8)
            img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
            if img is not None:
                return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        except Exception:
            pass
        return None

    def load_depth(self, session_id: str, frame_id: int) -> np.ndarray | None:
        """Loads depth map either from local path or GCP bucket."""
        # 1. Try local first
        if self.local_raw_path:
            # Check float16.gz first
            local_path_gz = self.local_raw_path / session_id / f"camera_{self.camera}" / self.view / "depth_maps" / f"{frame_id:06d}.float16.gz"
            if local_path_gz.exists():
                return self._parse_float16_gz(local_path_gz)
            # Try npz
            local_path_npz = self.local_raw_path / session_id / f"camera_{self.camera}" / self.view / "zed_depth_maps" / f"{frame_id:06d}.npz"
            if local_path_npz.exists():
                return self._parse_npz(local_path_npz)
        
        # 2. Try GCS anonymous download
        # Check npz first (ZED format)
        npz_path = f"{self.prefix}/{session_id}/camera_{self.camera}/{self.view}/zed_depth_maps/{frame_id:06d}.npz"
        try:
            blob = self.bucket.blob(npz_path)
            raw = blob.download_as_bytes(timeout=10)
            return self._parse_npz(io.BytesIO(raw))
        except Exception:
            pass

        # Check float16.gz fallback
        cre_path = f"{self.prefix}/{session_id}/camera_{self.camera}/{self.view}/depth_maps/{frame_id:06d}.float16.gz"
        try:
            blob = self.bucket.blob(cre_path)
            raw = blob.download_as_bytes(timeout=10)
            return self._parse_float16_gz_bytes(raw)
        except Exception:
            pass
        
        return None

    def _parse_npz(self, file_source) -> np.ndarray | None:
        try:
            depth = np.load(file_source)["arr_0"].astype(np.float32)
            depth[~np.isfinite(depth)] = 0
            depth[depth > 100] = 0
            return depth
        except Exception:
            return None

    def _parse_float16_gz(self, path: Path) -> np.ndarray | None:
        try:
            with gzip.open(path, 'rb') as f:
                return self._parse_float16_gz_bytes(f.read())
        except Exception:
            return None

    def _parse_float16_gz_bytes(self, data: bytes) -> np.ndarray | None:
        try:
            raw = np.frombuffer(gzip.decompress(data) if data.startswith(b'\x1f\x8b') else data, dtype=np.float16)
            # Truncate to match typical sizes
            for h, w in [(1242, 2208), (720, 1280), (1080, 1920)]:
                if raw.size >= h * w:
                    if raw.size >= h*w + 2 and np.allclose(raw[:2], 0):
                        return raw[2:h*w+2].reshape(h, w).astype(np.float32)
                    return raw[:h*w].reshape(h, w).astype(np.float32)
            side = int(np.sqrt(raw.size))
            return raw[:side*side].reshape(side, side).astype(np.float32)
        except Exception:
            return None

    def get_session_metadata(self, session_id: str) -> dict:
        path = f"{self.prefix}/{session_id}/description.json"
        blob = self.bucket.blob(path)
        try:
            return json.loads(blob.download_as_string(timeout=10))
        except Exception:
            return {}

# Initialize loader with project raw directory if present
SANPO_LOCAL_DIR = REPO_ROOT / "data" / "sanpo" / "raw"
loader = SANPOLoader(local_raw_path=SANPO_LOCAL_DIR, camera="head", view="left")
print("✅ SANPOLoader retrieval helper initialized.")


In [ ]:
# ── YOLO Model Setup ──────────────────────────────────────────────────────────

# Navigation-relevant classes from yolo_tracker.py
ALLOWED_CLASSES = {
    "person", "bicycle", "car", "motorcycle", "bus", "truck",
    "dog", "cat", "traffic light", "stop sign", "umbrella",
    "backpack", "suitcase",
}

print(f"Loading YOLO model from: {YOLO_MODEL_PATH} ...")
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
model = YOLO(YOLO_MODEL_PATH)
model.to(device)

print(f"\n================ MODEL SUMMARY ================")
print(f"Model Name:          {Path(YOLO_MODEL_PATH).name}")
print(f"Device:              {device.upper()}")

try:
    params = sum(p.numel() for p in model.model.parameters()) if hasattr(model, "model") else "Unknown"
    print(f"Total Parameters:    {params:,}" if isinstance(params, int) else f"Total Parameters:    {params}")
except Exception:
    print("Total Parameters:    Not inspectable")

supported_classes = list(model.names.values())
print(f"Supported Classes ({len(supported_classes)} total):")
print(f"  {', '.join(supported_classes[:10])} ...")
print(f"Confidence Threshold: {CONF_THRESH}")
print(f"IoU Threshold:        {IOU_THRESH}")
print(f"Class Whitelist ({len(ALLOWED_CLASSES)}):")
print(f"  {sorted(list(ALLOWED_CLASSES))}")
print(f"===============================================")


In [ ]:
# ── Robust Depth Gap Detection ────────────────────────────────────────────────

def find_depth_gaps(depth_map: np.ndarray,
                    yolo_dets: list[dict],
                    frame_h: int, frame_w: int) -> list[dict]:
    """
    Finds depth regions within DEPTH_MIN_M..DEPTH_MAX_M that have no
    corresponding YOLO bounding box.
    
    Uses Gaussian blurring and morphological operations to clean up
    sensor noise and isolate physical objects.
    """
    dh, dw = depth_map.shape[:2]

    # 1. Create a binary mask: True where depth is in target alert range
    alert_mask = (depth_map >= DEPTH_MIN_M) & (depth_map <= DEPTH_MAX_M)

    # 2. Mask out pixels already covered by YOLO bounding boxes (with scaling)
    scale_x = dw / frame_w
    scale_y = dh / frame_h
    yolo_mask = np.zeros_like(alert_mask)
    for det in yolo_dets:
        dx1 = int(det["x1"] * scale_x)
        dy1 = int(det["y1"] * scale_y)
        dx2 = int(det["x2"] * scale_x)
        dy2 = int(det["y2"] * scale_y)
        
        # Pad by 2px to mask out potential boundary bleed
        pad = 2
        dy1_pad = max(0, dy1 - pad)
        dy2_pad = min(dh, dy2 + pad)
        dx1_pad = max(0, dx1 - pad)
        dx2_pad = min(dw, dx2 + pad)
        yolo_mask[dy1_pad:dy2_pad, dx1_pad:dx2_pad] = True

    alert_mask[yolo_mask] = False

    # 3. Focus on navigation-relevant zone (lower 60% of frame)
    nav_start_row = int(dh * 0.4)
    nav_mask = np.zeros_like(alert_mask)
    nav_mask[nav_start_row:, :] = alert_mask[nav_start_row:, :]

    # Convert to uint8 for cv2 operations
    mask_u8 = (nav_mask * 255).astype(np.uint8)

    # 4. Noise suppression & cleaning
    blurred = cv2.GaussianBlur(mask_u8, (5, 5), 0)
    _, cleaned = cv2.threshold(blurred, 127, 255, cv2.THRESH_BINARY)

    # Morphological closing (fill small internal holes) followed by opening (eliminate tiny islands)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, kernel)
    cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_OPEN, kernel)

    # 5. Connected component contour analysis
    contours, _ = cv2.findContours(cleaned, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    blobs = []
    for i, cnt in enumerate(contours):
        area = cv2.contourArea(cnt)
        if area < MIN_BLOB_AREA_PX:
            continue

        # Centroid
        M = cv2.moments(cnt)
        if M["m00"] > 0:
            cx_dm = int(M["m10"] / M["m00"])
            cy_dm = int(M["m01"] / M["m00"])
        else:
            bx, by, bw, bh = cv2.boundingRect(cnt)
            cx_dm, cy_dm = bx + bw // 2, by + bh // 2

        # Extract depth measurements specifically from this contour region
        contour_mask = np.zeros_like(cleaned)
        cv2.drawContours(contour_mask, [cnt], -1, 255, -1)
        depths = depth_map[contour_mask == 255]
        depths = depths[(depths >= DEPTH_MIN_M) & (depths <= DEPTH_MAX_M)]

        if len(depths) == 0:
            continue

        mean_depth = float(np.mean(depths))
        min_depth = float(np.min(depths))

        # Bounding rectangle details
        rx, ry, rw, rh = cv2.boundingRect(cnt)
        aspect_ratio = float(rw) / max(float(rh), 1e-5)

        # Solidity (area / convex hull area)
        hull = cv2.convexHull(cnt)
        hull_area = cv2.contourArea(hull)
        solidity = float(area / hull_area) if hull_area > 0 else 0.0

        # Extent (area / bounding box area)
        extent = float(area / (rw * rh)) if (rw * rh) > 0 else 0.0

        # Elevation classification
        rel_y = cy_dm / dh
        if rel_y < 0.45:
            elevation = "head_level"
        elif rel_y < 0.65:
            elevation = "mid_level"
        else:
            elevation = "foot_level"

        # Approximate physical dimensions using head-camera metrics (~70deg HFOV)
        focal_length_px = 960.0 * (dw / 1280.0)
        est_width_m = (rw * mean_depth) / focal_length_px
        est_height_m = (rh * mean_depth) / focal_length_px

        blobs.append({
            "region_id": len(blobs) + 1,
            "cx_dm": cx_dm, "cy_dm": cy_dm,
            "cx_rgb": int(cx_dm / scale_x),
            "cy_rgb": int(cy_dm / scale_y),
            "area_px": int(area),
            "min_depth": min_depth,
            "mean_depth": mean_depth,
            "bbox_dm": [rx, ry, rx+rw, ry+rh],
            "bbox_rgb": [int(rx/scale_x), int(ry/scale_y), int((rx+rw)/scale_x), int((ry+rh)/scale_y)],
            "aspect_ratio": aspect_ratio,
            "solidity": solidity,
            "extent": extent,
            "elevation": elevation,
            "est_width_m": est_width_m,
            "est_height_m": est_height_m,
            "contour": cnt.tolist()
        })

    return sorted(blobs, key=lambda b: b["mean_depth"])

print("✅ Depth Gap Detector ready.")


In [ ]:
# ── Visualization Helpers ─────────────────────────────────────────────────────

ELEVATION_COLORS = {
    "head_level": (255, 50,  50),   # Red   — head/face height hazards
    "mid_level":  (255, 165,  0),   # Orange — waist height hazards
    "foot_level": (50,  200, 50),   # Green — trip hazards
}
YOLO_BOX_COLOR  = (0, 191, 255)     # Deep Sky Blue
YOLO_TEXT_COLOR = (255, 255, 255)

def visualize_frame_5panel(rgb_image: np.ndarray,
                           depth_map: np.ndarray,
                           yolo_dets: list[dict],
                           depth_blobs: list[dict],
                           save_path: Path | None = None,
                           show: bool = True):
    """
    Renders a high-quality 5-panel layout:
      1. Original RGB
      2. YOLO Detections overlay
      3. Depth Heatmap (Turbo map 0-10m)
      4. Extract Gaps mask
      5. Combined Overlay with labels and outlines
    """
    frame_h, frame_w = rgb_image.shape[:2]
    
    # P1: Original RGB
    p1_rgb = rgb_image.copy()
    
    # P2: YOLO Detections
    p2_yolo = rgb_image.copy()
    for det in yolo_dets:
        cv2.rectangle(p2_yolo, (det["x1"], det["y1"]), (det["x2"], det["y2"]), YOLO_BOX_COLOR, 3)
        label = f"{det['class_name']} {det['conf']:.2f}"
        cv2.putText(p2_yolo, label, (det["x1"], max(det["y1"]-8, 15)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, YOLO_BOX_COLOR, 2)
                    
    # P3: Depth Heatmap
    depth_clipped = np.clip(depth_map, 0.0, 10.0)
    depth_norm = (depth_clipped / 10.0 * 255).astype(np.uint8)
    depth_heatmap = cv2.applyColorMap(depth_norm, cv2.COLORMAP_TURBO)
    depth_heatmap = cv2.cvtColor(depth_heatmap, cv2.COLOR_BGR2RGB)
    p3_depth = cv2.resize(depth_heatmap, (frame_w, frame_h), interpolation=cv2.INTER_NEAREST)
    
    # P4: Gaps mask
    p4_gaps = np.zeros((frame_h, frame_w, 3), dtype=np.uint8)
    for blob in depth_blobs:
        color = ELEVATION_COLORS.get(blob["elevation"], (200, 200, 200))
        cnt = np.array(blob["contour"])
        scale_x = frame_w / depth_map.shape[1]
        scale_y = frame_h / depth_map.shape[0]
        cnt_scaled = cnt.copy()
        cnt_scaled[:, 0, 0] = (cnt[:, 0, 0] * scale_x).astype(int)
        cnt_scaled[:, 0, 1] = (cnt[:, 0, 1] * scale_y).astype(int)
        cv2.drawContours(p4_gaps, [cnt_scaled], -1, color, -1)
        
    # P5: Combined Overlay
    p5_overlay = rgb_image.copy()
    for det in yolo_dets:
        cv2.rectangle(p5_overlay, (det["x1"], det["y1"]), (det["x2"], det["y2"]), YOLO_BOX_COLOR, 2)
        label = f"{det['class_name']} {det['conf']:.2f}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(p5_overlay, (det["x1"], det["y1"]-th-4), (det["x1"], det["y1"]), YOLO_BOX_COLOR, -1)
        cv2.putText(p5_overlay, label, (det["x1"]+2, det["y1"]-2),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, YOLO_TEXT_COLOR, 1, cv2.LINE_AA)
                    
    for blob in depth_blobs:
        color = ELEVATION_COLORS.get(blob["elevation"], (200, 200, 200))
        cnt = np.array(blob["contour"])
        scale_x = frame_w / depth_map.shape[1]
        scale_y = frame_h / depth_map.shape[0]
        cnt_scaled = cnt.copy()
        cnt_scaled[:, 0, 0] = (cnt[:, 0, 0] * scale_x).astype(int)
        cnt_scaled[:, 0, 1] = (cnt[:, 0, 1] * scale_y).astype(int)
        
        # Outline
        cv2.drawContours(p5_overlay, [cnt_scaled], -1, color, 2)
        
        # Centroid
        cv2.circle(p5_overlay, (blob["cx_rgb"], blob["cy_rgb"]), 5, (255, 255, 255), -1)
        cv2.circle(p5_overlay, (blob["cx_rgb"], blob["cy_rgb"]), 6, color, 1)
        
        # Annotation box
        gap_label = f"GAP {blob['region_id']} ({blob['mean_depth']:.1f}m)"
        (tw, th), _ = cv2.getTextSize(gap_label, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
        cv2.rectangle(p5_overlay, (blob["cx_rgb"]-tw//2-2, blob["cy_rgb"]-th-10),
                      (blob["cx_rgb"]+tw//2+2, blob["cy_rgb"]-6), color, -1)
        cv2.putText(p5_overlay, gap_label, (blob["cx_rgb"]-tw//2, blob["cy_rgb"]-8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1, cv2.LINE_AA)
                    
    fig, axes = plt.subplots(1, 5, figsize=(25, 5))
    axes[0].imshow(p1_rgb)
    axes[0].set_title("1. Original RGB", fontsize=11, fontweight="bold")
    axes[0].axis("off")
    
    axes[1].imshow(p2_yolo)
    axes[1].set_title("2. YOLO Detections", fontsize=11, fontweight="bold")
    axes[1].axis("off")
    
    axes[2].imshow(p3_depth)
    axes[2].set_title("3. Depth Map (0-10m)", fontsize=11, fontweight="bold")
    axes[2].axis("off")
    
    axes[3].imshow(p4_gaps)
    axes[3].set_title("4. Depth-only Gaps", fontsize=11, fontweight="bold")
    axes[3].axis("off")
    
    axes[4].imshow(p5_overlay)
    axes[4].set_title("5. Combined Overlay", fontsize=11, fontweight="bold")
    axes[4].axis("off")
    
    legend_patches = [
        mpatches.Patch(color=np.array(YOLO_BOX_COLOR)/255.0, label="YOLO Bounding Box"),
        mpatches.Patch(color=np.array(ELEVATION_COLORS["head_level"])/255.0, label="Gap: Head level"),
        mpatches.Patch(color=np.array(ELEVATION_COLORS["mid_level"])/255.0, label="Gap: Mid level"),
        mpatches.Patch(color=np.array(ELEVATION_COLORS["foot_level"])/255.0, label="Gap: Foot level"),
    ]
    fig.legend(handles=legend_patches, loc="lower center", ncol=4, fontsize=10, bbox_to_anchor=(0.5, -0.05))
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches="tight")
    if show and SHOW_PLOTS_IN_NOTEBOOK:
        plt.show()
    plt.close()

print("✅ Visualisation utilities ready.")


In [ ]:
# ── Main Frame Processing Loop ────────────────────────────────────────────────

# Load stream definitions
with open(REPO_ROOT / "valid_streams.json") as f:
    valid_streams = json.load(f)

if SESSION_FILTER is not None:
    streams_to_process = [s for s in valid_streams if s["session_id"] in SESSION_FILTER]
else:
    streams_to_process = valid_streams[:MAX_STREAMS]

print(f"Loaded {len(valid_streams)} valid streams total. Processing subset of {len(streams_to_process)} streams.")

all_yolo_detections = []
all_candidate_regions = []
all_frame_stats = []

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

for idx, stream in enumerate(streams_to_process):
    session_id = stream["session_id"]
    camera = stream["camera"]
    view = stream["view"]
    
    print(f"\n📂 [{idx+1}/{len(streams_to_process)}] Processing Stream: {session_id} ({camera}/{view})")
    
    metadata = loader.get_session_metadata(session_id)
    
    # Detect available frame range
    available_frames = []
    local_frame_dir = SANPO_LOCAL_DIR / session_id / f"camera_{camera}" / view / "video_frames"
    if local_frame_dir.exists():
        available_frames = sorted([int(p.stem) for p in local_frame_dir.glob("*.png")])
    
    if not available_frames:
        available_frames = list(range(0, 100, 5))
        
    if not available_frames:
        print(f"  ⚠️ No frames discoverable for {session_id}.")
        continue
        
    if MAX_FRAMES_PER_STREAM and len(available_frames) > MAX_FRAMES_PER_STREAM:
        sampled_indices = np.linspace(0, len(available_frames)-1, MAX_FRAMES_PER_STREAM, dtype=int)
        sampled_frames = [available_frames[i] for i in sampled_indices]
    else:
        sampled_frames = available_frames

    print(f"   - Sampled {len(sampled_frames)} frames out of {len(available_frames)} available.")
    
    processed_count = 0
    for frame_id in sampled_frames:
        rgb = loader.load_rgb(session_id, frame_id)
        depth = loader.load_depth(session_id, frame_id)
        
        if rgb is None or depth is None:
            continue
            
        fh, fw = rgb.shape[:2]
        
        # YOLO Inference
        results = model.predict(cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR), conf=CONF_THRESH, iou=IOU_THRESH, verbose=False)[0]
        
        yolo_dets = []
        if results.boxes is not None:
            for box, cls_idx, conf in zip(
                results.boxes.xyxy.cpu().numpy(),
                results.boxes.cls.cpu().numpy().astype(int),
                results.boxes.conf.cpu().numpy(),
            ):
                cls_name = results.names[cls_idx]
                if cls_name not in ALLOWED_CLASSES:
                    continue
                x1, y1, x2, y2 = map(int, box)
                yolo_dets.append({
                    "stream_id": session_id,
                    "frame_id": frame_id,
                    "class_name": cls_name,
                    "conf": float(conf),
                    "x1": x1, "y1": y1, "x2": x2, "y2": y2,
                    "area_px": (x2 - x1) * (y2 - y1),
                    "aspect_ratio": (x2 - x1) / max((y2 - y1), 1)
                })
        
        all_yolo_detections.extend(yolo_dets)
        
        # Depth gaps analysis
        blobs = find_depth_gaps(depth, yolo_dets, fh, fw)
        
        for blob in blobs:
            all_candidate_regions.append({
                "stream_id": session_id,
                "frame_id": frame_id,
                "region_id": blob["region_id"],
                "area": blob["area_px"],
                "centroid_x": blob["cx_rgb"],
                "centroid_y": blob["cy_rgb"],
                "mean_depth": blob["mean_depth"],
                "minimum_depth": blob["min_depth"],
                "aspect_ratio": blob["aspect_ratio"],
                "solidity": blob["solidity"],
                "extent": blob["extent"],
                "elevation": blob["elevation"],
                "bounding_box": blob["bbox_rgb"],
                "estimated_size_w": blob["est_width_m"],
                "estimated_size_h": blob["est_height_m"]
            })
            
        # Compute frame metrics
        nav_area = 0.60 * fh * fw
        yolo_nav_mask = np.zeros((fh, fw), dtype=bool)
        nav_start_row = int(fh * 0.4)
        for det in yolo_dets:
            dy1 = max(nav_start_row, det["y1"])
            dy2 = max(nav_start_row, det["y2"])
            dx1 = det["x1"]
            dx2 = det["x2"]
            if dy2 > dy1 and dx2 > dx1:
                yolo_nav_mask[dy1:dy2, dx1:dx2] = True
        
        yolo_nav_px = np.sum(yolo_nav_mask)
        coverage_pct = (yolo_nav_px / nav_area) * 100.0
        
        total_gap_area = sum(b["area_px"] for b in blobs)
        largest_gap_area = max((b["area_px"] for b in blobs), default=0)
        avg_conf = np.mean([d["conf"] for d in yolo_dets]) if yolo_dets else 0.0
        
        frame_stat = {
            "stream_id": session_id,
            "frame_id": frame_id,
            "detection_count": len(yolo_dets),
            "candidate_region_count": len(blobs),
            "total_candidate_area": total_gap_area,
            "largest_candidate_area": largest_gap_area,
            "coverage_percentage": coverage_pct,
            "unexplained_percentage": 100.0 - coverage_pct,
            "average_confidence": avg_conf,
            "notes": ""
        }
        
        # Hard Example Mining selection
        is_hard = False
        notes = []
        if largest_gap_area > 3000:
            is_hard = True
            notes.append("Large gap")
        if len(blobs) >= 3:
            is_hard = True
            notes.append("Multi-gap obstacle")
        if len(yolo_dets) > 0 and avg_conf < 0.45 and len(blobs) > 0:
            is_hard = True
            notes.append("Low confidence overlay")
            
        frame_stat["notes"] = ", ".join(notes) if notes else "Normal"
        all_frame_stats.append(frame_stat)
        
        # Save Visualisation if needed
        save_vis_path = None
        if len(blobs) > 0 and SAVE_VISUALIZATIONS:
            save_vis_path = OUTPUT_DIR / f"{session_id}_{frame_id:06d}_gap_vis.png"
            
        # Show first frame visualization only to prevent notebook clutter
        show_vis = (processed_count == 0)
        visualize_frame_5panel(rgb, depth, yolo_dets, blobs, save_path=save_vis_path, show=show_vis)
        
        # Save raw frame to hard examples folder if qualified
        if is_hard and EXPORT_HARD_EXAMPLES:
            hard_save_path = HARD_EXAMPLES_DIR / f"hard_{session_id}_{frame_id:06d}.png"
            cv2.imwrite(str(hard_save_path), cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))
            
        processed_count += 1
        
    print(f"   Completed. Processed {processed_count} frames successfully.")

print("\n✅ All streams processed successfully.")


In [ ]:
# ── Detection Statistics ──────────────────────────────────────────────────────

if all_yolo_detections:
    df_yolo = pd.DataFrame(all_yolo_detections)
    
    print("\n════════════════════════ YOLO DETECTION STATS ════════════════════════")
    print(f"Total whitelisted detections: {len(df_yolo)}")
    
    print("\nDetections per Class:")
    print(df_yolo["class_name"].value_counts().to_string())
    
    print("\nAverage Confidence per Class:")
    print(df_yolo.groupby("class_name")["conf"].mean().to_string())
    
    # Display size-based distributions
    largest = df_yolo.sort_values(by="area_px", ascending=False).head(5)
    smallest = df_yolo.sort_values(by="area_px", ascending=True).head(5)
    
    print("\nLargest detections (Top 5):")
    print(largest[["stream_id", "frame_id", "class_name", "area_px", "conf"]].to_string(index=False))
    
    print("\nSmallest detections (Top 5):")
    print(smallest[["stream_id", "frame_id", "class_name", "area_px", "conf"]].to_string(index=False))
    
    # Confidence histogram and size distribution plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].hist(df_yolo["conf"], bins=15, color='skyblue', edgecolor='black')
    axes[0].set_title("Confidence Distribution of YOLO Hits")
    axes[0].set_xlabel("Confidence Score")
    axes[0].set_ylabel("Count")
    
    axes[1].hist(df_yolo["area_px"], bins=15, color='salmon', edgecolor='black')
    axes[1].set_title("Bounding Box Pixel Area Distribution")
    axes[1].set_xlabel("Bbox Area (px)")
    axes[1].set_ylabel("Count")
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ No YOLO detections found to evaluate.")


In [ ]:
# ── Gap Statistics ────────────────────────────────────────────────────────────

if all_candidate_regions:
    df_gaps = pd.DataFrame(all_candidate_regions)
    df_frames = pd.DataFrame(all_frame_stats)
    
    print("\n════════════════════════ DEPTH GAP STATS ════════════════════════")
    print(f"Total Missed Regions (Gaps):      {len(df_gaps)}")
    print(f"Average Region Pixel Area:        {df_gaps['area'].mean():.1f} px")
    print(f"Median Region Pixel Area:         {df_gaps['area'].median():.1f} px")
    print(f"Largest Missed Obstacle Area:     {df_gaps['area'].max()} px")
    
    print("\nGaps by Elevation Level:")
    print(df_gaps["elevation"].value_counts().to_string())
    
    print("\nDepth Range Distribution (meters):")
    print(df_gaps["mean_depth"].describe()[["mean", "min", "50%", "max"]].to_string())
    
    print("\nGaps Frequency per Stream:")
    print(df_gaps.groupby("stream_id").size().to_string())

    mean_unexplained = df_frames["unexplained_percentage"].mean()
    print(f"\nAverage Navigation Area Unexplained by YOLO: {mean_unexplained:.2f}%")
    
    # Plots
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].hist(df_gaps["mean_depth"], bins=10, color='lightgreen', edgecolor='black')
    axes[0].set_title("Depth Distribution of Gaps")
    axes[0].set_xlabel("Distance (m)")
    axes[0].set_ylabel("Count")
    
    axes[1].hist(df_gaps["area"], bins=15, color='orange', edgecolor='black')
    axes[1].set_title("Missed Gaps Pixel Size Distribution")
    axes[1].set_xlabel("Gap Area (px)")
    axes[1].set_ylabel("Count")
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ No depth gaps identified in the current stream run.")


In [ ]:
# ── Failure Review ────────────────────────────────────────────────────────────

if all_frame_stats:
    df_frames = pd.DataFrame(all_frame_stats)
    
    print("\n════════════════════════ FAILURE REVIEW TOOL ════════════════════════")
    print("Top 10 Frames with Largest Unexplained Obstacle Areas (Potential Hazard Misses):")
    top_misses = df_frames.sort_values(by="total_candidate_area", ascending=False).head(10)
    print(top_misses[["stream_id", "frame_id", "detection_count", "candidate_region_count", 
                     "total_candidate_area", "coverage_percentage", "notes"]].to_string(index=False))
    
    complete_misses = df_frames[(df_frames["detection_count"] == 0) & (df_frames["candidate_region_count"] > 0)]
    if not complete_misses.empty:
        print("\nFrames where YOLO missed everything but Depth detected objects (Complete Misses):")
        print(complete_misses.sort_values(by="candidate_region_count", ascending=False).head(10)[
            ["stream_id", "frame_id", "candidate_region_count", "total_candidate_area", "notes"]
        ].to_string(index=False))
else:
    print("⚠️ No frame-level stats available for failure review.")


In [ ]:
# ── CSV Export ────────────────────────────────────────────────────────────────

if EXPORT_CSV and all_frame_stats:
    frame_csv_path = OUTPUT_DIR / "frame_summary.csv"
    df_f = pd.DataFrame(all_frame_stats)
    df_f.to_csv(frame_csv_path, index=False)
    print(f"✅ Saved Frame Summary CSV: {frame_csv_path}")
    
    regions_csv_path = OUTPUT_DIR / "candidate_regions.csv"
    if all_candidate_regions:
        df_r = pd.DataFrame(all_candidate_regions)
        df_r["bounding_box"] = df_r["bounding_box"].apply(lambda b: f"{b[0]};{b[1]};{b[2]};{b[3]}")
        df_r.to_csv(regions_csv_path, index=False)
        print(f"✅ Saved Candidate Regions CSV: {regions_csv_path}")
    else:
        print("ℹ️ No regions to output to CSV.")


In [ ]:
# ── Fine-Tuning Recommendations Report ────────────────────────────────────────
from IPython.display import display, Markdown

report_lines = []
report_lines.append("# Fine-Tuning & Dataset Gap Analysis Report")
report_lines.append(f"**Generated on:** {time.strftime('%Y-%m-%d %H:%M:%S')}")
report_lines.append("")

# Section 1
report_lines.append("## 1. Detection Performance")
if all_yolo_detections:
    df_y = pd.DataFrame(all_yolo_detections)
    class_counts = df_y["class_name"].value_counts()
    class_conf = df_y.groupby("class_name")["conf"].mean()
    strongest_class = class_conf.idxmax()
    weakest_class = class_conf.idxmin()
    
    report_lines.append(f"* **Most Common Detections:** `{class_counts.index[0]}` ({class_counts.values[0]} hits)")
    report_lines.append(f"* **Strongest Class (Highest Mean Confidence):** `{strongest_class}` ({class_conf[strongest_class]:.2f})")
    report_lines.append(f"* **Weakest Class (Lowest Mean Confidence):** `{weakest_class}` ({class_conf[weakest_class]:.2f})")
else:
    report_lines.append("* No YOLO detections occurred. Cannot evaluate strongest/weakest classes.")
report_lines.append("")

# Section 2
report_lines.append("## 2. Depth Gap Analysis")
if all_candidate_regions:
    df_g = pd.DataFrame(all_candidate_regions)
    elev_counts = df_g["elevation"].value_counts()
    most_common_elev = elev_counts.index[0] if not elev_counts.empty else "None"
    
    close_gaps = len(df_g[df_g["mean_depth"] <= 2.0])
    mid_gaps = len(df_g[(df_g["mean_depth"] > 2.0) & (df_g["mean_depth"] <= 4.0)])
    far_gaps = len(df_g[df_g["mean_depth"] > 4.0])
    
    report_lines.append(f"* **Total Uncovered Gaps:** {len(df_g)}")
    report_lines.append(f"* **Most Common Elevation of Missed Objects:** `{most_common_elev}`")
    report_lines.append(f"* **Missed Gaps by Proximity:**")
    report_lines.append(f"  * **Immediate hazard zone (<= 2m):** {close_gaps} instances (Critical risk)")
    report_lines.append(f"  * **Mid-range navigation (2m - 4m):** {mid_gaps} instances")
    report_lines.append(f"  * **Far-range space (> 4m):** {far_gaps} instances")
else:
    report_lines.append("* No depth gaps detected.")
report_lines.append("")

# Section 3
report_lines.append("## 3. Dataset & Fine-Tuning Recommendations")
recs = []
if all_candidate_regions:
    df_g = pd.DataFrame(all_candidate_regions)
    elev_counts = df_g["elevation"].value_counts()
    total_gaps = len(df_g)
    
    if elev_counts.get("foot_level", 0) > total_gaps * 0.4:
        recs.append("1. **Prioritize Ground-Level Hazard Dataset Curation:** A large proportion of gaps are detected at ground/foot level. We need more annotations for steps, potholes, curb changes, and ground trash. Use data augmentation that focuses on low angles/ground planes.")
    if elev_counts.get("head_level", 0) > total_gaps * 0.15:
        recs.append("2. **Collect Head-Height Obstacles:** Found substantial head-level depth gaps (e.g. low pipes, tree branches, hanging signs). Introduce a `hanging_hazard` class during the next fine-tuning cycle to capture these high-risk areas.")
    if elev_counts.get("mid_level", 0) > total_gaps * 0.3:
        recs.append("3. **Street Infrastructure Expansion:** Mid-level gaps indicate street infrastructure like bollards, poles, concrete bases, or barriers. Introduce a concrete class `bollard` to improve classification accuracy and localization.")

if all_yolo_detections:
    df_y = pd.DataFrame(all_yolo_detections)
    class_conf = df_y.groupby("class_name")["conf"].mean()
    weak_classes = class_conf[class_conf < 0.50].index.tolist()
    if weak_classes:
        recs.append(f"4. **Target Low-Confidence Classes:** Boost whitelisted classes with low confidence (<50%) by adding more target images: {', '.join([f'`{c}`' for c in weak_classes])}.")

recs.append("5. **Environment Augmentations:** Introduce shadow training patterns (to prevent false depth regions due to contrast transitions) and motion-blur overlays mimicking typical visually impaired head scan behaviors.")
recs.append("6. **Annotation Curation:** Review exported frames in `hard_examples/` to verify boundaries, annotate previously missed items, and update annotations for classes like `person` or `dog` in cluttered contexts.")

for r in recs:
    report_lines.append(r)
report_lines.append("")

# Priority action plan table
report_lines.append("## 4. Prioritized Action Plan")
report_lines.append("| Priority | Action Item | Target / Details |")
report_lines.append("|---|---|---|")
report_lines.append("| 🔴 **CRITICAL** | Mine Hard Examples | Verify and annotate frames saved in `hard_examples/`. |")
if all_yolo_detections and 'weakest_class' in locals() and class_conf[weakest_class] < 0.45:
    report_lines.append(f"| 🟠 **HIGH** | Boost Confidence | Target dataset collection (500+ labels) for weak class `{weakest_class}`. |")
else:
    report_lines.append("| 🟠 **HIGH** | Curation for Gaps | Introduce `bollard` class to resolve mid-level structural gap misses. |")
report_lines.append("| 🟡 **MEDIUM** | Shadow & Blur Augmentation | Implement exposure/shadow contrast variations in PyTorch training pipeline. |")

report_text = "\n".join(report_lines)
report_path = OUTPUT_DIR / "fine_tuning_recommendations.md"
with open(report_path, "w") as f:
    f.write(report_text)
print(f"✅ Saved recommendations report to: {report_path}\n")

display(Markdown(report_text))
